# run_e01_e04
CPU analysis worker for Stage 4 E01-E04. Run cells 2, 3 and 4 in order. Configure only cell 3.

- Set `MODEL = 'e03'` once to compute the shared E03 absolute and pooled-context-mean unit (`E03/e03/shared`).
- Set `MODEL = 'qwen'` or `MODEL = 'gemma'` separately on independent CPU runtimes to compute that model's E01, E02 (fixed depths + image-only visual), E03 neural-vs-shallow comparisons (reusing shared E03 predictions), and E04 pooling-sensitivity units.
- Each completed unit is checkpointed immediately via `publish_analysis_block` and its exact immutable completion reference is streamed to stdout. Preserve those references for resume (`RESUME_REFS` / `RESUME_ANALYSES_FILE`) and for `validate_stage4.ipynb`.
- Optional E04 arms (`post_caption`, `decoder_visual_pool`) remain unscheduled and are recorded as `unavailable` in the output summary and final Stage 4 release.


In [ ]:
import os, sys, subprocess
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
from google.colab import drive
drive.mount('/content/drive')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO = Path('/content/Research-stage4-e01-e04')
CODE_REF = 'phase1'  # Resolved once below; set to the approved Stage 4 integration commit to pin explicitly.
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-stage2-cpu.txt')], check=True)
print('Fixed code commit:', CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
import json
MODEL = 'qwen'  # 'qwen', 'gemma', or 'e03' (run 'e03' once; run 'qwen' and 'gemma' separately).
STAGE4_PLAN = PHASE1_ROOT / 'REPLACE_STAGE4_PLAN/_COMPLETE.json'
# JSON file containing canonical prediction completion references (list, {block_id: ref}, or {'predictions': [...]}).
# For MODEL='qwen' or 'gemma', include both that model's SCREEN predictions and the shared E03 predictions.
PREDICTIONS_FILE = PHASE1_ROOT / 'REPLACE_STAGE4_PREDICTIONS.json'
EXPERIMENTS = ['E01', 'E02', 'E03', 'E04']  # Optional subset (for MODEL='e03', ['E03'] or default).
UNITS = []  # Optional explicit unit IDs, e.g. ['E01/qwen/18']; leave empty for all units of MODEL.
RESUME_ANALYSES_FILE = None  # Optional Path to prior completed analyses JSON for verified resume.
RESUME_REFS = []  # Optional list of explicit {'path': ..., 'sha256': ...} analysis completion refs.
OUTPUT_INDEX = None  # Optional Path under PHASE1_ROOT to save the completed E01-E04 unit index JSON.


In [ ]:
if MODEL not in ('qwen', 'gemma', 'e03'):
    raise RuntimeError(f'Unsupported MODEL selector: {MODEL}')
if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Stage 4 checkout.')
for path in (STAGE4_PLAN, PREDICTIONS_FILE):
    if not Path(path).is_file():
        raise RuntimeError(f'Set the explicit mounted artifact in cell 3: {path}')
command = [
    sys.executable, '-u', '-m', 'src.stage4.run_e01_e04',
    '--phase1-root', str(PHASE1_ROOT),
    '--manifest', str(STAGE4_PLAN),
    '--owner', MODEL,
    '--predictions', str(PREDICTIONS_FILE),
    '--storage-scope', 'persistent-drive',
]
if EXPERIMENTS:
    command += ['--experiments', *EXPERIMENTS]
for unit_id in UNITS:
    command += ['--unit', unit_id]
if RESUME_ANALYSES_FILE is not None:
    if not Path(RESUME_ANALYSES_FILE).is_file():
        raise RuntimeError(f'Missing resume analyses file: {RESUME_ANALYSES_FILE}')
    command += ['--resume-analyses', str(RESUME_ANALYSES_FILE)]
for ref in RESUME_REFS:
    command += ['--resume-ref', json.dumps(ref)]
if OUTPUT_INDEX is not None:
    command += ['--output-index', str(OUTPUT_INDEX)]
process = subprocess.Popen(command, cwd=REPO, env=os.environ.copy(), stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    import signal
    process.send_signal(signal.SIGINT)
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    raise
if process.wait() != 0:
    raise RuntimeError('E01-E04 worker blocked/failed. Preserve streamed references before resuming.')
